
# 練習問題: ファイル入出力

## 目標

ファイルを開いて読み込み, 終端まで繰り返し読むループの書き方を身につける.

## 課題

`fileio.cpp` (または `fileio.f90`) は, まず `data.txt` に 5 行の数値 (`i` と `i*0.5`) を書き出す (この部分は完成済み).
そのあと `data.txt` を読み直して, 2列目 `x` の合計を求めたい.
読み込みループが空なので, 現状の合計は `0`.

`TODO` の箇所に **1行ずつ読み, 読めた間 `x` を `sum` に足し込むループ** を書け.

- C++: `while (fscanf(rp, "%d %lf", &i, &x) == 2) { sum += x; }` (2個読めた間繰り返す)
- Fortran: `do` … `read(10, *, iostat=ios) i, x` … `if (ios /= 0) exit` … `sum = sum + x` … `end do` (`iostat` が 0 でなくなったら終端)

## コンパイルと実行

```
# C++
nvc++ -fast fileio.cpp -o fileio.exe

# Fortran
nvfortran -fast fileio.f90 -o fileio.exe
```

```
./fileio.exe
cat data.txt   # 書き出された中身を確認
```

## 期待される結果

`x` は `0.0, 0.5, 1.0, 1.5, 2.0` なので合計は 5.0.

```
sum of x = 5.000000
```

読み込みループを書く前は `0` になる. `data.txt` の中身も `cat` で確認せよ.



# 1. AIチューター
- 以下は必要に応じて実行（毎度実行する必要はない）


In [ ]:
import heytutor


## 1-1. 一般的な質問
- ChatGPTなどに聞くときのように自由に質問可能。
- ただし「答えを教えて」などは自制すること。


In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 1-2. この問題に関するヒント
- `{file:problem.md}` は上記の問題文に展開される。


In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}


## 1-3. いくつかの変数
* それぞれ以下のように展開される。

* `{file:FILENAME}` : _FILENAME_ の中身
* `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前の入力・出力, etc.

## 1-4. 困ったときのヘルプ
* コンパイル時や実行時のエラー直後に以下を実行するとエラーに関するヘルプが得られる。


In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:fileio.cpp}

コマンドと実行結果:
{bash[-1]}



## 1-5. フィードバック
* 答えが出た後も、無駄なところや、より良いやり方がないかを聞くことを推奨。
* 以下のファイル名は適宜書き換えよ (Fortranなら `.cpp` -> `.f90` とするなど)


In [ ]:
%%hey

フィードバックを下さい。

問題:
{file:problem.md}

私の答:
{file:fileio.cpp}


# 2. ジョブ投入ツール
- 以下を実行しておくと、`%%bash_submit_a` (Aquariousへのジョブ投入), `%%bash_submit_o` (Odyssey へのジョブ投入) が使えるようになる


In [ ]:
import wisteria_submit

# 3. C++ ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ fileio.cpp
#include <cstdio>

int main() {
  /* まず data.txt に 0..4 の i と i*0.5 を書き出す (この部分は完成済み) */
  FILE * wp = fopen("data.txt", "w");
  if (wp == NULL) { printf("cannot open for write\n"); return 1; }
  for (int i = 0; i < 5; i++) {
    fprintf(wp, "%d %f\n", i, i * 0.5);
  }
  fclose(wp);

  /* 次に data.txt を読み直し, 2列目 (x) の合計を求める */
  FILE * rp = fopen("data.txt", "r");
  if (rp == NULL) { printf("cannot open for read\n"); return 1; }
  int i;
  double x;
  double sum = 0.0;
  // TODO: fscanf で1行ずつ (i, x) を読み, x を sum に足し込むループを書け.
  fclose(rp);
  printf("sum of x = %f\n", sum);
  return 0;
}

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore fileio.cpp -o fileio_cpp.exe

## 3-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./fileio_cpp.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a

./fileio_cpp.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./fileio_cpp.exe

## 3-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:fileio.cpp}


# 4. Fortran ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ fileio.f90
program fileio
  integer :: i, ios
  real(8) :: x, sum
  ! まず data.txt に 0..4 の i と i*0.5 を書き出す (この部分は完成済み)
  open(unit=10, file="data.txt", status="replace", action="write")
  do i = 0, 4
     write(10, "(i0,1x,f0.6)") i, i * 0.5d0
  end do
  close(10)

  ! 次に data.txt を読み直し, 2列目 (x) の合計を求める
  open(unit=10, file="data.txt", status="old", action="read")
  sum = 0.0d0
  ! TODO: read で1行ずつ (i, x) を読み (iostat で終端判定), x を sum に足し込むループを書け.
  close(10)
  print "(a,f0.6)", "sum of x = ", sum
end program fileio

## 4-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore fileio.f90 -o fileio_f90.exe

## 4-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./fileio_f90.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a
./fileio_f90.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit

#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./fileio_f90.exe

## 4-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:fileio.f90}